In [ ]:
import pandas as pd

from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

import numpy as np
from sklearn import preprocessing
from sklearn import utils
from sklearn.metrics import mean_squared_error
from math import sqrt
from sklearn.metrics import r2_score

import matplotlib.pyplot as plt
import seaborn as sns

import time
import sys
#import pickle
#from sklearn.externals import joblib

In [ ]:
fpath = '../data/CRU/station/'
stnpath = '../data/MODIS/'

stmm = 1
enmm = 1
mon = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 
       'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

h = [9,10,11,12,
     12,13,13,14,14,15,15,
     16,16,16,17,17,17,18,18,
     18,19,19,19,20,20,21,
     21,22,22,23,23,24,
     25,26]

v = [2,2,2,1,
     2,1,2,1,2,1,2,
     0,1,2,0,1,2,0,1,
     2,0,1,2,1,2,1,
     2,1,2,1,2,2,
     2,2]

for mm in np.arange(stmm,enmm+1):
    print(mm)
    
    train = pd.read_csv(fpath + repr(mm) + '_train_new.csv')
    print(train.shape)
    train = train.set_index("STN") # STN을 Index로 지정
    train_year = train["YEAR"]
    train = train[["LAT", "LON",
                   "NDVI_t", "LST_DAY_t", "LST_NIGHT_t",
                   "NDVI_a", "LST_DAY_a", "LST_NIGHT_a",
                   "Water", "Evergreen_trees", "Decidous_trees",
                   "Shrub", "Grass", "crops", "Urban", "Snow",
                   "Barren", "Unclassifiend", "SAT"]]
    
    label_name = "SAT"
    feature_names = train.columns.difference([label_name])


    
# Train
    X_train = train[feature_names]
    y_train = train[label_name] 

    
#################
# Random Forest #
#################
    RF = RandomForestRegressor(n_estimators = 1000,  
                                   n_jobs = -1,
                                   random_state = 42)
    RF.fit(X_train, y_train)

#################
### LightGBM ###
#################
    LGBM = LGBMRegressor(n_estimators = 1000,  
                        learning_rate = 0.1)#,
#                       max_depth = 6)

    LGBM.fit(X_train, y_train)
    
#################
#### XGBoost ####
#################   
    XGB = XGBRegressor(n_estimators = 1000, 
                       learning_rate = 0.1)#,
#                      max_depth = 6)

    XGB.fit(X_train, y_train)
    
    

##### set year info
    if mm <= 2:
        styy = 2001
        enyy = 2022
        nyr  = enyy - styy + 1
    if mm == 3:
        styy = 2000
        enyy = 2022
        nyr  = enyy - styy + 1
    if mm == 4:
        styy = 2000
        enyy = 2022
        nyr  = enyy - styy + 1
    if mm > 4:
        styy = 2000
        enyy = 2021
        nyr  = enyy - styy + 1
    nyr = int(nyr)
    npix = int(1200)
    
    for t in np.arange(0,34):
        hh = h[t]
        vv = v[t]
        print(hh,vv)
             
#        dat = np.load(stnpath + repr(mm).zfill(2) + '/h' + repr(hh).zfill(2) + 'v' + repr(vv).zfill(2)+'/h'+ repr(hh).zfill(2) + 'v' + repr(vv).zfill(2)+'_dat.npy')
        dat = np.load(stnpath + repr(mm).zfill(2) + '/h'+ repr(hh).zfill(2) + 'v' + repr(vv).zfill(2)+'_dat.npy')
        test = dat[:,0:7,:,:]
        
        for yy in np.arange(styy,enyy+1):
            start = time.time()
            ii = yy-styy
            print(ii)
            dat1=dat[ii,:,:,:]
            dat1=dat1.reshape(18,npix*npix)
            da = np.transpose(dat1)
            da = np.nan_to_num(da)
            dat2 = pd.DataFrame(da,columns=["LAT", "LON", "NDVI_t", "NDVI_a",
                                            "LST_DAY_t", "LST_DAY_a", "LST_NIGHT_t", "LST_NIGHT_a",
                                            "Water", "Evergreen_trees", "Decidous_trees",
                                            "Shrub", "Grass", "crops", "Urban", "Snow",
                                            "Barren", "Unclassifiend"])
            X_test = dat2[feature_names]
            X_test.fillna(0, inplace=True)
            print(X_test)
            y_test_predict = RF.predict(X_test)
            yy = y_test_predict.reshape(npix,npix)
            test[ii,2,:,:] = yy

            y_test_predict = LGBM.predict(X_test)
            yy = y_test_predict.reshape(npix,npix)
            test[ii,3,:,:] = yy

            y_test_predict = XGB.predict(X_test)
            yy = y_test_predict.reshape(npix,npix)
            test[ii,4,:,:] = yy
            
            test[ii,5,:,:] = (test[ii,2,:,:]+test[ii,3,:,:]+test[ii,4,:,:])/3.
            
   
        np.save(stnpath + repr(mm).zfill(2) + '/h'+ repr(hh).zfill(2) + 'v' + repr(vv).zfill(2)+'_sat.npy',test)
        print("time :", time.time() - start)